In [41]:
# imports
import pandas as pd
import boto3
from io import StringIO

In [42]:
# get data from s3
client = boto3.client("s3")
bucket = "reddit-data-306915831561-us-east-2-an"
files = client.list_objects_v2(Bucket=bucket)["Contents"]
reddit_data = pd.concat([
    pd.read_csv(StringIO(client.get_object(Bucket=bucket, Key=f["Key"])["Body"].read().decode("utf-8")))
    for f in files
])

In [43]:
# combine title and body into 1 column
def combine_text(row):
    if pd.isna(row["selftext"]) or row["selftext"] in ("", "[deleted]", "[removed]", "Title."):
        return row["title"]
    return row["title"] + " | " + row["selftext"]
reddit_data["text"] = reddit_data.apply(lambda x: combine_text(x), axis=1)

In [44]:
# drop columns
reddit_data.drop(columns=["created_utc", "num_comments", "score", "created_at", "selftext", "title"], inplace=True)
reddit_data.head()

,id,subreddit,year,month,text
0,abjdx2,careerguidance,2019,1,I don't want to be a support worker anymore. W...
1,abl1kb,careerguidance,2019,1,"what are the best, free online certification c..."
2,abo3m3,careerguidance,2019,1,Have any of you regretted fast tracking your c...
3,abpfqe,careerguidance,2019,1,I don't want to cook for a living anymore. Wha...
4,absbor,careerguidance,2019,1,Communications or Marketing majors... where ha...


In [57]:
# randomly sample data for use
sample_pct = 0.1
sampled_data = reddit_data.groupby("year", group_keys=False).apply(lambda x: x.sample(frac=sample_pct, random_state=699))
sampled_data.head()

/var/folders/mj/jdg6nf_91m30_bh08_9683mr0000gn/T/ipykernel_48935/2037045512.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_data = reddit_data.groupby("year", group_keys=False).apply(lambda x: x.sample(frac=sample_pct, random_state=699))


,id,subreddit,year,month,text
3284,e9gtb5,cscareerquestions,2019,12,Is career in defense a dead end? | I see all t...
979,bqwez0,jobs,2019,5,What do you do when your interviewer is passiv...
1948,djw9b4,jobs,2019,10,300 Job Applications Down And I’ve Finally Lan...
1460,cjveyp,jobs,2019,7,I got a job! | I got an email from British Col...
2021,cs3w2u,cscareerquestions,2019,8,When to start applying for summer internships?


In [58]:
data = reddit_data[~reddit_data.id.isin(sampled_data.id)]
annotate_data = data.groupby("year", group_keys=False).apply(lambda x: x.sample(min(len(x), 50), random_state=699))
annotate_data.head()

/var/folders/mj/jdg6nf_91m30_bh08_9683mr0000gn/T/ipykernel_48935/3501387277.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  annotate_data = data.groupby("year", group_keys=False).apply(lambda x: x.sample(min(len(x), 50), random_state=699))


,id,subreddit,year,month,text
1002,bs7mvu,jobs,2019,5,Why do hiring managers do this?
1779,d6imv9,jobs,2019,9,A simple way to make the job hunt a little mor...
1670,calny9,cscareerquestions,2019,7,So I just landed a work from home Software Dev...
3414,eei3jk,cscareerquestions,2019,12,Applied for grad SE role. Got bumped up | As t...
796,b8xk8s,cscareerquestions,2019,4,How to avoid making your coworkers life misera...


In [62]:
annotate_data.to_csv("annotate_data.csv", index=False)